# SocialLLM — Minimal Run (3 personas × 2 tasks × 1 episode)
Stripped-down version for fast iteration on CPU. Same pipeline as the full notebook.

## 0 · Configuration

In [1]:
# ── Edit these before running ──────────────────────────────────────────
MODEL          = "gpt2"
N_EPISODES     = 1            # 1 episode per condition
DEVICE         = "cpu"
TASKS          = ["qa", "story"]   # 2 tasks: factual + narrative
PERSONAS       = ["curious", "formal", "skeptical"]  # 3 most distinct styles
FDR_ALPHA      = 0.05
AGGREGATION    = "mean"
# ── Total: 3 × 2 × 1 = 6 interactions ──────────────────────────────────

import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

DATA_DIR         = "../data"
ACTIVATIONS_H5   = f"{DATA_DIR}/activations/activations_{MODEL.replace('-','_')}_minimal.h5"
INTERACTIONS_DIR = f"{DATA_DIR}/interactions_minimal"
FIGURES_DIR      = "../results/figures"

for d in [FIGURES_DIR, INTERACTIONS_DIR, f"{DATA_DIR}/activations"]:
    os.makedirs(d, exist_ok=True)

print(f"Model : {MODEL}")
print(f"Device: {DEVICE}")
print(f"Personas: {PERSONAS}")
print(f"Tasks   : {TASKS}")
print(f"Conditions: {len(PERSONAS)} personas × {len(TASKS)} tasks × {N_EPISODES} episode = "
      f"{len(PERSONAS) * len(TASKS) * N_EPISODES} interactions")

Model : gpt2
Device: cpu
Personas: ['curious', 'formal', 'skeptical']
Tasks   : ['qa', 'story']
Conditions: 3 personas × 2 tasks × 1 episode = 6 interactions


## 1 · Imports

In [2]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.social.agents    import LLMAgent, TargetAgent, make_interlocutor_configs, make_target_config
from src.social.environment import InteractionDatasetRunner
from src.social.tasks     import TASK_REGISTRY
from src.activation.extractor import ActivationDataset, load_layer, list_layers
from src.analysis.selectivity import run_full_analysis
from src.analysis.decoding    import decode_layer, pca_population_geometry, compute_rdm

print(f"PyTorch {torch.__version__} | device={DEVICE}")

PyTorch 2.5.1 | device=cpu


## 2 · Social Interactions

In [ ]:
# Load target
print(f"Loading target ({MODEL})...")
target = TargetAgent(make_target_config(device=DEVICE, model_name=MODEL))
print("Target ready.")

# Load only the 3 selected personas (curious=0, formal=1, skeptical=4)
all_cfgs = make_interlocutor_configs(device=DEVICE, model_name=MODEL)
cfg_map  = {cfg.agent_id.replace("persona_", ""): cfg for cfg in all_cfgs}
interlocutors = [LLMAgent(cfg_map[p]) for p in PERSONAS]
print(f"Interlocutors: {[a.agent_id for a in interlocutors]}")

Loading target (gpt2)...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 17273.96it/s]


Target ready.


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 17360.43it/s]

Interlocutors: ['persona_curious', 'persona_formal', 'persona_skeptical']


: 

In [ ]:
import random, json
from pathlib import Path
from IPython.display import display, HTML, clear_output
from src.social.environment import SocialEnvironment

tasks = [TASK_REGISTRY[t] for t in TASKS]

# ── colour scheme for the live chat panel ────────────────────────────────
_P_COLORS = {"curious": "#005493", "formal": "#B30000", "skeptical": "#7B2D8E"}
_T_COLOR  = "#2e7d32"

def _chat_html(interaction) -> str:
    persona = interaction.interlocutor_id.replace("persona_", "")
    pc = _P_COLORS.get(persona, "#555")
    rows = ""
    for t in interaction.turns:
        is_tgt = (t.speaker == interaction.target_id)
        bg     = "#e8f5e9" if is_tgt else "#eef2ff"
        lc     = _T_COLOR  if is_tgt else pc
        lbl    = "TARGET"  if is_tgt else persona.upper()
        rows += (
            f'<div style="margin:5px 0;padding:8px 10px;background:{bg};border-radius:6px">'
            f'<span style="color:{lc};font-weight:bold;font-size:.78em">{lbl}</span><br>'
            f'<span style="font-size:.88em;white-space:pre-wrap">{t.content}</span></div>'
        )
    return (
        f'<div style="border:1.5px solid {pc};border-radius:8px;margin:10px 0;'
        f'overflow:hidden;font-family:sans-serif">'
        f'<div style="background:{pc};color:white;padding:6px 14px;font-weight:bold">'
        f'{persona} × {interaction.task_id}</div>'
        f'<div style="padding:10px 14px">{rows}</div></div>'
    )

# ── run loop ─────────────────────────────────────────────────────────────
interactions = []
chat_panels  = []
rng          = random.Random(42)
total_runs   = len(interlocutors) * len(tasks) * N_EPISODES
out_dir      = Path(INTERACTIONS_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

for interlocutor in interlocutors:
    for task in tasks:
        env = SocialEnvironment(target=target, interlocutor=interlocutor, task=task, rng=rng)
        for ep in range(N_EPISODES):
            n = len(interactions) + 1
            persona = interlocutor.agent_id.replace("persona_", "")
            clear_output(wait=True)
            display(HTML(
                f'<b style="font-family:sans-serif">⏳ [{n}/{total_runs}] running '
                f'<span style="color:{_P_COLORS.get(persona,"#555")}">{persona}</span>'
                f' × {task.task_id} …</b>'
                + "".join(chat_panels)
            ))

            iid = f"{task.task_id}__{interlocutor.agent_id}__ep{ep:03d}"
            interaction = env.run_episode(interaction_id=iid, verbose=False)
            interactions.append(interaction)

            # save JSON transcript
            (out_dir / f"{iid}.json").write_text(
                json.dumps(interaction.to_dict(), indent=2)
            )

            chat_panels.append(_chat_html(interaction))

# ── final display ─────────────────────────────────────────────────────────
clear_output(wait=True)
display(HTML(
    f'<b style="font-family:sans-serif;color:{_T_COLOR}">✓ {len(interactions)}/{total_runs} interactions completed</b>'
    + "".join(chat_panels)
))
print(f"\nTotal interactions collected: {len(interactions)}")

In [ ]:
ds = ActivationDataset(output_path=ACTIVATIONS_H5, aggregation=AGGREGATION)
ds.add_from_interactions(interactions)
ds.save()

layers = list_layers(ACTIVATIONS_H5)
data0, labels0, label_map = load_layer(ACTIVATIONS_H5, layers[0])

print(f"Saved {len(layers)} layers → {ACTIVATIONS_H5}")
print(f"First layer shape : {data0.shape}  (N_samples × hidden_dim)")
print(f"Label map         : {label_map}")

residual_layers = sorted([l for l in layers if "residual"   in l])
mlp_layers      = sorted([l for l in layers if "mlp_hidden" in l])
persona_names   = [label_map[str(i)] for i in range(len(label_map))]
persona_short   = [n.replace("persona_", "") for n in persona_names]
COLORS          = ["#005493", "#B30000", "#7B2D8E"]  # blue / red / purple

## 3 · Selectivity Analysis

In [ ]:
print("Running selectivity analysis across all layers...")
reports = run_full_analysis(ACTIVATIONS_H5, fdr_alpha=FDR_ALPHA)
print(f"Done — {len(reports)} layers analyzed.\n")

mid = residual_layers[len(residual_layers)//2]
r   = reports[mid]
print(f"--- {mid} ---")
print(f"  Fraction selective (FDR {FDR_ALPHA}): {r['fraction_selective']:.3f}")
print(f"  Mean SI                            : {r['mean_si']:.3f}")
print(f"  Mean d-prime                       : {r['mean_dprime']:.3f}")

### Figure 1 — Fraction selective per layer

In [ ]:
import seaborn as sns
sns.set_style("whitegrid")

frac_res = [reports[l]["fraction_selective"] for l in residual_layers]
frac_mlp = [reports[l]["fraction_selective"] for l in mlp_layers]
x = range(len(residual_layers))

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.plot(x, frac_res, "o-",  color="#005493", label="Residual stream", lw=2)
ax.plot(x, frac_mlp, "s--", color="#B30000", label="MLP hidden",      lw=2)
ax.axhline(FDR_ALPHA, color="#6B6B6B", ls=":", lw=1.5, label=f"FDR α={FDR_ALPHA}")
ax.set_xlabel("Layer index", fontsize=12, fontweight="bold")
ax.set_ylabel("Fraction selective", fontsize=12, fontweight="bold")
ax.set_title("Fraction of Persona-Selective Units per Layer", fontsize=13, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(range(len(residual_layers)))
ax.legend()
ax.grid(linestyle="--", linewidth=0.6, alpha=0.7)
plt.savefig(f"{FIGURES_DIR}/min_fig1_fraction_selective.png", dpi=150, bbox_inches="tight")
plt.savefig(f"{FIGURES_DIR}/min_fig1_fraction_selective.pdf", bbox_inches="tight")
plt.show()

### Figure 2 — Tuning curves of top selective units

In [ ]:
best_sel_layer = max(reports, key=lambda l: reports[l]["fraction_selective"])
report = reports[best_sel_layer]
si_scores = report["selectivity_indices"]
top6      = np.argsort(si_scores)[::-1][:6]
cmeans    = report["condition_means"]

fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
for idx, unit in enumerate(top6):
    ax   = axes[idx // 3][idx % 3]
    vals = cmeans[:, unit]
    ax.bar(range(len(persona_short)), vals, color=COLORS, alpha=0.85, edgecolor="white")
    ax.set_xticks(range(len(persona_short)))
    ax.set_xticklabels(persona_short, rotation=30, ha="right", fontsize=9)
    ax.set_title(f"Unit {unit}  |  SI={si_scores[unit]:.3f}", fontsize=10)
    ax.set_ylabel("Mean activation" if idx % 3 == 0 else "")
    ax.axhline(0, color="black", lw=0.5)
    ax.grid(linestyle="--", linewidth=0.6, alpha=0.7, axis="y")

plt.suptitle(f"Tuning Curves — Top 6 Selective Units\n({best_sel_layer})",
             fontsize=13, fontweight="bold")
plt.savefig(f"{FIGURES_DIR}/min_fig2_tuning_curves.png", dpi=150, bbox_inches="tight")
plt.savefig(f"{FIGURES_DIR}/min_fig2_tuning_curves.pdf", bbox_inches="tight")
plt.show()

## 4 · Population Decoding

In [ ]:
print("Running linear decoding across all layers...")
decoding_results = {}
for layer_name in sorted(reports):
    d, lbl, lm = load_layer(ACTIVATIONS_H5, layer_name)
    res = decode_layer(d, lbl, n_folds=3)   # 3-fold for small N
    res["label_map"] = lm
    decoding_results[layer_name] = res
    print(f"  {layer_name:40s}  acc={res['mean_accuracy']:.3f}")
print("Done.")

### Figure 3 — Decoding accuracy + PCA geometry

In [ ]:
res_layers_d = sorted([l for l in decoding_results if "residual"   in l])
mlp_layers_d = sorted([l for l in decoding_results if "mlp_hidden" in l])
chance       = decoding_results[res_layers_d[0]]["chance_level"]
x            = range(len(res_layers_d))

acc_res = [decoding_results[l]["mean_accuracy"] for l in res_layers_d]
acc_mlp = [decoding_results[l]["mean_accuracy"] for l in mlp_layers_d]

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ax.plot(x, acc_res, "o-",  color="#005493", label="Residual stream", lw=2)
ax.plot(x, acc_mlp, "s--", color="#B30000", label="MLP hidden",      lw=2)
ax.axhline(chance, color="#6B6B6B", ls=":", lw=1.5, label=f"Chance ({chance:.2f})")
ax.set_xlabel("Layer index", fontsize=12, fontweight="bold")
ax.set_ylabel("Decoding accuracy (3-fold CV)", fontsize=12, fontweight="bold")
ax.set_title("Linear Decoding of Interlocutor Persona Across Layers", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.05); ax.legend()
ax.set_xticks(x); ax.set_xticklabels(range(len(res_layers_d)))
ax.grid(linestyle="--", linewidth=0.6, alpha=0.7)
plt.savefig(f"{FIGURES_DIR}/min_fig3_decoding.png", dpi=150, bbox_inches="tight")
plt.savefig(f"{FIGURES_DIR}/min_fig3_decoding.pdf", bbox_inches="tight")
plt.show()

best_layer_dec = max(decoding_results, key=lambda l: decoding_results[l]["mean_accuracy"])
print(f"Best layer: {best_layer_dec}  acc={decoding_results[best_layer_dec]['mean_accuracy']:.3f}")

In [ ]:
# PCA geometry at best decoding layer
data_pca, labels_pca, _ = load_layer(ACTIVATIONS_H5, best_layer_dec)
projected, var_ratio     = pca_population_geometry(data_pca, labels_pca)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, (d1, d2) in zip(axes, [(0,1),(0,2)]):
    for i, (name, col) in enumerate(zip(persona_short, COLORS)):
        mask = labels_pca == i
        ax.scatter(projected[mask,d1], projected[mask,d2],
                   c=col, label=name, alpha=0.8, s=80,
                   edgecolors="white", linewidths=0.5)
    ax.set_xlabel(f"PC{d1+1} ({var_ratio[d1]*100:.1f}%)", fontsize=11, fontweight="bold")
    ax.set_ylabel(f"PC{d2+1} ({var_ratio[d2]*100:.1f}%)", fontsize=11, fontweight="bold")
    ax.legend(title="Persona", fontsize=9)
    ax.grid(linestyle="--", linewidth=0.6, alpha=0.7)
axes[0].set_title("PC1 vs PC2", fontsize=12, fontweight="bold")
axes[1].set_title("PC1 vs PC3", fontsize=12, fontweight="bold")
plt.suptitle(f"Population Geometry (PCA) — {best_layer_dec}", fontsize=13, fontweight="bold")
plt.savefig(f"{FIGURES_DIR}/min_fig4_pca.png", dpi=150, bbox_inches="tight")
plt.savefig(f"{FIGURES_DIR}/min_fig4_pca.pdf", bbox_inches="tight")
plt.show()

## 5 · RSA — RDMs across 3 sampled layers

In [ ]:
n_show      = 3
step        = max(1, len(residual_layers) // n_show)
show_layers = residual_layers[::step][:n_show]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, layer_name in zip(axes, show_layers):
    means = reports[layer_name]["condition_means"]
    rdm   = compute_rdm(means, metric="correlation")
    im    = ax.imshow(rdm, cmap="RdYlBu_r", vmin=0, vmax=rdm.max())
    ax.set_xticks(range(len(persona_short))); ax.set_yticks(range(len(persona_short)))
    ax.set_xticklabels(persona_short, rotation=40, ha="right", fontsize=9)
    ax.set_yticklabels(persona_short, fontsize=9)
    lshort = (layer_name.replace("block_","L")
                        .replace("_residual","-res")
                        .replace("_mlp_hidden","-mlp"))
    ax.set_title(lshort, fontsize=10, fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Representational Dissimilarity Matrices (Correlation Distance)",
             fontsize=13, fontweight="bold")
plt.savefig(f"{FIGURES_DIR}/min_fig5_rdms.png", dpi=150, bbox_inches="tight")
plt.savefig(f"{FIGURES_DIR}/min_fig5_rdms.pdf", bbox_inches="tight")
plt.show()

## 6 · Summary

In [ ]:
best_dec = max(decoding_results, key=lambda l: decoding_results[l]["mean_accuracy"])
best_sel = max(reports,          key=lambda l: reports[l]["fraction_selective"])
n_above  = sum(1 for r in decoding_results.values() if r["above_chance"])

print("=" * 55)
print("  SOCIALLLM — MINIMAL RUN SUMMARY")
print("=" * 55)
print(f"  Model             : {MODEL}")
print(f"  Personas          : {PERSONAS}")
print(f"  Tasks             : {TASKS}")
print(f"  Episodes/cond     : {N_EPISODES}")
print(f"  Total interactions: {len(interactions)}")
print()
print(f"  [SELECTIVITY]")
print(f"    Most selective layer : {best_sel}")
print(f"    Fraction selective   : {reports[best_sel]['fraction_selective']:.1%}")
print(f"    Mean SI              : {reports[best_sel]['mean_si']:.3f}")
print()
print(f"  [DECODING]")
print(f"    Best layer           : {best_dec}")
print(f"    Peak accuracy        : {decoding_results[best_dec]['mean_accuracy']:.1%}")
print(f"    Chance               : {decoding_results[best_dec]['chance_level']:.1%}")
print(f"    Above-chance layers  : {n_above}/{len(decoding_results)}")
print("=" * 55)